# prefopt-repro: a small run on real data

SFT → reward model → DPO on **Qwen2.5-0.5B-Instruct** with real
**UltraFeedback** pairs, then win-rate against the SFT baseline judged by an
**independent** reward model (Skywork-Reward-V2-Qwen3-0.6B, never trained
here), plus the reward-hacking diagnostics.

This is a small run, not the ablation grid: one base model, one β, a few
thousand pairs, 150 evaluation prompts. It measures what this pipeline does at
that scale; it does not compare DPO against GRPO.

**How to run**

1. Colab: *Runtime → Change runtime type → T4 GPU* (any GPU works; A100/L4 run in bf16).
   Kaggle also works: *Settings → Accelerator → GPU*, *Internet on*.
2. *Runtime → Run all*. Roughly 1–2 h on a free T4. Keep the tab open.
3. The last cell downloads `small_run_results.zip`.

Set `USE_DRIVE = True` below to keep run directories on Google Drive, so a
disconnected session resumes from the last finished stage instead of starting over.

In [ ]:
import os, subprocess, sys, time, json
import torch

assert torch.cuda.is_available(), "No GPU. Runtime > Change runtime type > T4 GPU."
GPU = torch.cuda.get_device_name(0)
CAP = torch.cuda.get_device_capability(0)
NATIVE_BF16 = CAP[0] >= 8          # T4 / P100 are older and run in float32
DTYPE = "bfloat16" if NATIVE_BF16 else "float32"
print(f"{GPU} (compute {CAP[0]}.{CAP[1]}) -> {DTYPE}")

In [ ]:
USE_DRIVE = False   # True: keep runs/ on Google Drive so a disconnect can resume

HOME = os.getcwd()
REPO = os.path.join(HOME, "prefopt-repro")
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "https://github.com/Ghazaleh-Ramezani/prefopt-repro.git", REPO], check=True)
os.chdir(REPO)
SHA = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"]).decode().strip()
print("prefopt-repro at", SHA)

subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "transformers==4.57.6", "trl==0.20.0", "peft==0.21.2",
                "datasets==5.0.1", "accelerate==1.15.0", "sentencepiece"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", "."], check=True)

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    RUNS = "/content/drive/MyDrive/prefopt_small_run/runs"
else:
    RUNS = os.path.join(REPO, "runs/small_run")
os.makedirs(RUNS, exist_ok=True)
print("runs ->", RUNS)

## Settings

LoRA learning rates are higher than the repo defaults, which are full-fine-tuning
values. With a few dozen optimiser steps per stage, the defaults barely move a
LoRA adapter.

In [ ]:
MODEL   = "Qwen/Qwen2.5-0.5B-Instruct"
JUDGE   = "Skywork/Skywork-Reward-V2-Qwen3-0.6B"   # independent: not trained on anything here
DATASET = "HuggingFaceH4/ultrafeedback_binarized"
SFT_SPLIT, TRAIN_SPLIT, EVAL_SPLIT = "train_sft", "train_prefs", "test_prefs"
MAX_LEN, MAX_PROMPT = 1024, 512

N_SFT      = 1000    # SFT examples (train_sft)
N_RM_PAIRS = 2000    # reward-model pairs (train_prefs)
N_DPO      = 1000    # DPO pairs: the first 1000 of the same shuffled pairs
N_EVAL     = 500     # held-out pairs (test_prefs) for RM accuracy
N_PROMPTS  = 150     # held-out prompts for win-rate and diagnostics
SEEDS      = 2       # generation seeds for win-rate
BETA       = 0.1

COMMON = [
    "--set", f"model.base_model={MODEL}",
    "--set", "model.attn_implementation=sdpa",
    "--set", f"model.torch_dtype={DTYPE}",
    "--set", f"train.bf16={'true' if NATIVE_BF16 else 'false'}",
    "--set", f"data.dataset={DATASET}",
    "--set", f"data.sft_train_split={SFT_SPLIT}",
    "--set", f"data.train_split={TRAIN_SPLIT}",
    "--set", f"data.eval_split={EVAL_SPLIT}",
    "--set", f"data.max_length={MAX_LEN}",
    "--set", f"data.max_prompt_length={MAX_PROMPT}",
    "--set", f"data.max_eval_samples={N_EVAL}",
    "--set", "train.eval_strategy=no",     # one evaluation at the end of each stage
]
TIMINGS = {}

def run(name, args, done_file):
    # Runs one stage with live output; skipped when its output already exists.
    if os.path.exists(done_file):
        print(f"[{name}] already done, skipping")
        return
    print(f"[{name}] starting")
    t0 = time.time()
    proc = subprocess.Popen([sys.executable, "-m", *args], stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end="")
    if proc.wait() != 0:
        raise RuntimeError(f"[{name}] failed with exit code {proc.returncode}")
    TIMINGS[name] = round((time.time() - t0) / 60, 1)
    print(f"[{name}] done in {TIMINGS[name]} min")

## 1. SFT baseline

In [ ]:
run("sft", ["prefopt.train_sft", "--config", "configs/sft.yaml", *COMMON,
            "--set", f"data.max_train_samples={N_SFT}",
            "--set", "train.learning_rate=1e-4",
            "--set", "train.per_device_train_batch_size=2",
            "--set", "train.gradient_accumulation_steps=8",
            "--set", "name=sft", "--output-dir", f"{RUNS}/sft"],
    f"{RUNS}/sft/adapter_config.json")

## 2. Reward model

In [ ]:
run("rm", ["prefopt.train_rm", "--config", "configs/rm.yaml", *COMMON,
           "--set", f"data.max_train_samples={N_RM_PAIRS}",
           "--set", "train.learning_rate=1e-4",
           "--set", "train.per_device_train_batch_size=4",
           "--set", "train.gradient_accumulation_steps=4",
           "--set", "name=rm", "--output-dir", f"{RUNS}/rm"],
    f"{RUNS}/rm/rm_eval.json")
print(json.dumps(json.load(open(f"{RUNS}/rm/rm_eval.json")), indent=2))

## 3. DPO

If this cell runs out of GPU memory, change the batch size to 1 and gradient
accumulation to 16 below; the effective batch stays 16.

In [ ]:
run("dpo", ["prefopt.train_dpo", "--config", "configs/dpo.yaml", *COMMON,
            "--set", f"init_adapter={RUNS}/sft",
            "--set", f"data.max_train_samples={N_DPO}",
            "--set", "data.max_eval_samples=200",
            "--set", "train.learning_rate=5e-5",
            "--set", f"method.beta={BETA}",
            "--set", "train.per_device_train_batch_size=2",
            "--set", "train.gradient_accumulation_steps=8",
            "--set", f"name=dpo_b{BETA}", "--output-dir", f"{RUNS}/dpo_b{BETA}"],
    f"{RUNS}/dpo_b{BETA}/dpo_eval.json")
print(json.dumps({k: v for k, v in json.load(open(f"{RUNS}/dpo_b{BETA}/dpo_eval.json")).items()
                  if "reward" in k or "loss" in k}, indent=2))

## 4. Win-rate vs SFT, judged by the independent reward model

Position-swapped, bootstrap 95% CI, two generation seeds. A reward-model judge
scores each response on its own, so it cannot be position-biased and swap
consistency is 1.0 by construction.

In [ ]:
POLICY = f"{RUNS}/dpo_b{BETA}"
run("eval", ["prefopt.evaluate", "--config", "configs/dpo.yaml", *COMMON,
             "--policy", POLICY, "--baseline", f"{RUNS}/sft",
             "--judge", "rm", "--rm-path", JUDGE,
             "--n", str(N_PROMPTS), "--seeds", str(SEEDS)],
    f"{POLICY}/results.json")

## 5. Reward-hacking diagnostics under the proxy reward model

KL from the SFT reference, length–reward correlation, distinct-4 and stylistic
tells, scored by the reward model trained in step 2 (the proxy).

In [ ]:
run("diagnostics", ["prefopt.diagnostics", "--config", "configs/dpo.yaml", *COMMON,
                    "--policy", POLICY, "--reference", f"{RUNS}/sft",
                    "--rm-path", f"{RUNS}/rm", "--n", str(N_PROMPTS)],
    f"{POLICY}/diagnostics.json")

## 6. Checks on the judges

On the same 500 held-out pairs the reward model was evaluated on:

* agreement of the independent judge with the UltraFeedback preference labels
  (these labels come from GPT-4 ratings, not humans);
* the trained reward model's accuracy through the evaluation harness's scoring
  path, which should match the trainer's own `eval_accuracy`;
* the same reward model scored on plain `prompt\n\nresponse` text, the input
  format the harness used before it rendered the chat template.

In [ ]:
import gc
from datasets import load_dataset
from prefopt.evaluate import RewardModelJudge

CHECK = f"{RUNS}/judge_checks.json"
if not os.path.exists(CHECK):
    pairs = load_dataset(DATASET, split=EVAL_SPLIT)
    pairs = pairs.shuffle(seed=17).select(range(N_EVAL))   # the RM's held-out pairs (data.py)
    rows = [(r["prompt"], r["chosen"][-1]["content"], r["rejected"][-1]["content"]) for r in pairs]

    def accuracy(judge, plain=False):
        if plain:   # the old input: no chat template
            judge.format = lambda prompt, response: f"{prompt}\n\n{response}"
        try:
            return sum(judge.score(p, c) > judge.score(p, r) for p, c, r in rows) / len(rows)
        finally:
            if plain:
                del judge.format

    t0 = time.time()
    checks = {"n_pairs": len(rows)}
    ext = RewardModelJudge(rm_path=JUDGE, base_model=MODEL)
    checks["independent_judge_label_agreement"] = accuracy(ext)
    del ext; gc.collect(); torch.cuda.empty_cache()

    rm = RewardModelJudge(rm_path=f"{RUNS}/rm", base_model=MODEL)
    checks["trained_rm_accuracy_chat_format"] = accuracy(rm)
    checks["trained_rm_accuracy_plain_format"] = accuracy(rm, plain=True)
    del rm; gc.collect(); torch.cuda.empty_cache()

    checks["trainer_eval_accuracy"] = json.load(open(f"{RUNS}/rm/rm_eval.json")).get("eval_accuracy")
    json.dump(checks, open(CHECK, "w"), indent=2)
    TIMINGS["judge_checks"] = round((time.time() - t0) / 60, 1)
print(json.dumps(json.load(open(CHECK)), indent=2))

## 7. Length filter

TRL's RewardTrainer drops pairs longer than `max_length` rather than truncating
them, so the reward model trained and was scored on fewer pairs than requested.

In [ ]:
from transformers import AutoTokenizer

LEN = f"{RUNS}/length_filter.json"
if not os.path.exists(LEN):
    tok = AutoTokenizer.from_pretrained(MODEL)
    def kept(split, n):
        ds = load_dataset(DATASET, split=split).shuffle(seed=17).select(range(n))
        ok = 0
        for r in ds:
            c = tok.apply_chat_template(r["chosen"], tokenize=False)
            j = tok.apply_chat_template(r["rejected"], tokenize=False)
            ok += len(tok(c).input_ids) <= MAX_LEN and len(tok(j).input_ids) <= MAX_LEN
        return {"requested": n, "kept": ok}
    json.dump({"train": kept(TRAIN_SPLIT, N_RM_PAIRS), "eval": kept(EVAL_SPLIT, N_EVAL),
               "max_length": MAX_LEN}, open(LEN, "w"), indent=2)
print(json.dumps(json.load(open(LEN)), indent=2))

## 8. Report and download

In [ ]:
import shutil, zipfile

subprocess.run([sys.executable, "-m", "prefopt.report", "--runs", RUNS,
                "--out", f"{RUNS}/RESULTS.md"], check=True)

res = json.load(open(f"{POLICY}/results.json"))
diag = json.load(open(f"{POLICY}/diagnostics.json"))
summary = {
    "git_sha": SHA, "gpu": GPU, "dtype": DTYPE, "model": MODEL, "judge": JUDGE,
    "n_sft": N_SFT, "n_rm_pairs": N_RM_PAIRS, "n_dpo_pairs": N_DPO, "beta": BETA,
    "n_prompts": N_PROMPTS, "seeds": SEEDS,
    "rm_eval_accuracy": json.load(open(f"{RUNS}/rm/rm_eval.json")).get("eval_accuracy"),
    "win_rate_mean": res["win_rate_mean"],
    "win_rate_per_seed": [(s["win_rate"], s["ci95"]) for s in res["per_seed"]],
    "proxy_rm_win_rate_seed0": diag["rm_judged_win_rate"],
    "goodhart_gap_seed0": diag["rm_judged_win_rate"] - res["per_seed"][0]["win_rate"],
    "kl_from_reference": diag.get("kl_from_reference"),
    "length_reward_corr": diag["length_reward_corr"],
    "mean_chars_policy": diag["style_policy"]["mean_chars"],
    "mean_chars_reference": diag["style_reference"]["mean_chars"],
    "minutes_per_stage": TIMINGS,
}
json.dump(summary, open(f"{RUNS}/summary.json", "w"), indent=2)
print(open(f"{RUNS}/RESULTS.md").read())
print(json.dumps(summary, indent=2))

# Text artifacts only; adapter weights stay in runs/.
OUT = os.path.join(HOME, "small_run_results")
shutil.rmtree(OUT, ignore_errors=True)
os.makedirs(OUT)
for f in ("RESULTS.md", "summary.json", "judge_checks.json", "length_filter.json"):
    shutil.copy(f"{RUNS}/{f}", OUT)
for run_dir in sorted(os.listdir(RUNS)):
    src = os.path.join(RUNS, run_dir)
    if not os.path.isdir(src):
        continue
    os.makedirs(os.path.join(OUT, run_dir), exist_ok=True)
    for f in ("config.resolved.yaml", "environment.json", "rm_eval.json", "dpo_eval.json",
              "results.json", "diagnostics.json"):
        if os.path.exists(os.path.join(src, f)):
            shutil.copy(os.path.join(src, f), os.path.join(OUT, run_dir))
ZIP = os.path.join(HOME, "small_run_results.zip")
with zipfile.ZipFile(ZIP, "w", zipfile.ZIP_DEFLATED) as z:
    for root, _, files in os.walk(OUT):
        for f in files:
            full = os.path.join(root, f)
            z.write(full, os.path.relpath(full, HOME))
print("->", ZIP)
try:
    from google.colab import files
    files.download(ZIP)
except ImportError:
    print("Not on Colab: download small_run_results.zip from the output panel.")